<a href="https://colab.research.google.com/github/fbeilstein/bioinformatics/blob/master/practice_06_cellular_state_spaces.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Cellular State Spaces (Manifolds & Differentiation)**

In Lecture 5 we turned a raw droplet matrix into a clean set of **individual cells**. Now we ask the next question: *what kinds of cells are they, and how are they related to one another?*

In 1957, Conrad Waddington pictured cell development as a ball rolling down a hilly **epigenetic landscape**. Valleys stand for stable cell fates (T cell, B cell, monocyte). Ridges are barriers between fates. Each cell is a point in a space with about 20,000 dimensions, one per gene. But gene regulatory networks limit which states a cell can actually reach. As a result, real cells sit on a **low-dimensional manifold** inside this huge space.

Our goal is to **learn that manifold from data**:
1.  **Spectral decomposition (PCA):** Compress about 2,000 noisy gene axes into about 40 informative axes.
2.  **$k$-Nearest Neighbor ($k$-NN) graph:** Link every cell to its closest transcriptional neighbors. The graph is a discrete approximation of the manifold.
3.  **Leiden community detection:** Split the graph into densely connected communities. These are candidate cell types.
4.  **UMAP:** Draw the graph in 2D so we can look at it.
5.  **Marker genes:** Use known biology to name each community (immune cell lineage determination).

We will use **Scanpy** (`sc.pp.pca`, `sc.pp.neighbors`, `sc.tl.umap`, `sc.tl.leiden`) on the PBMC3k dataset from Lecture 5.


#### **Part 1: Environment Setup**

Leiden clustering needs the `igraph` graph library (with `leidenalg` as an optional backend), so we install it alongside Scanpy.


In [ ]:
import os
from IPython.display import clear_output

# Install scanpy, anndata, graph libraries for Leiden, and standard libraries
!pip install -q scanpy anndata igraph leidenalg numpy pandas scipy scikit-learn matplotlib seaborn

clear_output()
print("Installations successful.")

#### **Part 2: Dataset Download**

We download the same **raw, unfiltered** PBMC3k matrix from 10x Genomics that we used in Lecture 5. This keeps the notebook self-contained: it works in a fresh Colab session.


In [ ]:
import urllib.request
import tarfile

import shutil

url = "https://cf.10xgenomics.com/samples/cell-exp/1.1.0/pbmc3k/pbmc3k_raw_gene_bc_matrices.tar.gz"
filename = "pbmc3k_raw_gene_bc_matrices.tar.gz"

if not os.path.exists("raw_gene_bc_matrices/hg19"):
    print("Downloading 10x Genomics PBMC3k raw dataset...")
    # The 10x CDN rejects the default 'Python-urllib' User-Agent (HTTP 403), so we send a browser-like one
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request) as response, open(filename, "wb") as out_file:
        shutil.copyfileobj(response, out_file)
    print("Extracting dataset...")
    with tarfile.open(filename, "r:gz") as tar:
        tar.extractall()

clear_output()
print("Raw data available in the 'raw_gene_bc_matrices/hg19' folder.")


#### **Part 3: Rebuilding the Lecture 5 Result (QC Filtering)**

We repeat the Lecture 5 pipeline in a few lines:
*   Remove empty droplets (≤ 500 UMIs).
*   Remove dying cells, i.e. cells whose mitochondrial % is more than 3 scaled MADs above the median.
*   Remove likely doublets (> 2,500 detected genes).
*   Remove genes detected in fewer than 3 cells. They carry no information about population structure.


In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=80, facecolor='white')

# Load the raw matrix
adata = sc.read_10x_mtx('raw_gene_bc_matrices/hg19/', var_names='gene_symbols', cache=True)
adata.var_names_make_unique()

# QC metrics
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

# 1. Empty droplets
adata = adata[adata.obs.total_counts > 500, :].copy()

# 2. Dying cells (dynamic MAD threshold)
pct_mt = adata.obs['pct_counts_mt'].values
median_mt = np.median(pct_mt)
mad_scaled = 1.4826 * np.median(np.abs(pct_mt - median_mt))
mt_threshold = median_mt + 3 * mad_scaled

# 3. Doublets
adata = adata[(adata.obs['pct_counts_mt'] < mt_threshold) &
              (adata.obs['n_genes_by_counts'] <= 2500), :].copy()

# 4. Uninformative genes
sc.pp.filter_genes(adata, min_cells=3)

print(f"MT threshold: {mt_threshold:.2f}%")
print(f"Clean matrix: {adata.n_obs:,} cells × {adata.n_vars:,} genes")


#### **Part 4: Normalization, Log-Transform and Highly Variable Genes**

Droplets capture different amounts of RNA, so raw counts mix up **biology** with **capture efficiency**. We apply three standard steps:

1.  **Library-size normalization:** Rescale each cell to 10,000 total counts (counts per 10k).
2.  **Log transform:** $x \mapsto \log(1 + x)$. This tames the heavy right tail of count data and makes the variance roughly independent of the mean.
3.  **Highly Variable Genes (HVGs):** Most genes are housekeeping genes, expressed at about the same level in every cell, so they carry no information about cell identity. We keep only the genes whose dispersion is unusually high for their mean expression.

We save the full log-normalized matrix in `adata.raw` before subsetting to HVGs. This lets us still look up *any* gene later, for example for marker plots.


In [ ]:
# Keep the raw counts in a layer
adata.layers['counts'] = adata.X.copy()

# Normalize and log-transform
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# Freeze the full log-normalized matrix (all genes) for later visualization
adata.raw = adata

# Highly variable genes
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
print(f"Highly variable genes: {adata.var.highly_variable.sum():,} / {adata.n_vars:,}")
sc.pl.highly_variable_genes(adata)

# Subset to HVGs and z-score each gene (clip extreme values at 10 SD)
adata = adata[:, adata.var.highly_variable].copy()
sc.pp.scale(adata, max_value=10)

print(f"Matrix entering PCA: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

#### **Part 5: Spectral Decomposition (PCA)**

Let $X \in \mathbb{R}^{n \times p}$ be the centered matrix ($n$ cells, $p$ genes). PCA finds orthogonal directions of maximal variance. These are the eigenvectors of the gene–gene covariance matrix:

$$ C = \frac{1}{n-1} X^T X = V \Lambda V^T $$

In practice we compute them via the **Singular Value Decomposition** $X = U S V^T$, without ever forming $C$:
*   The **principal axes (loadings)** are the columns of $V$. They are "meta-genes", i.e. weighted combinations of genes.
*   The **cell coordinates (scores)** are $U S$.
*   The **variance explained** by component $i$ is $\lambda_i = s_i^2 / (n-1)$.

Biological signal (co-regulated gene programs) spreads across many genes and ends up in the first few PCs. Technical noise is independent between genes and spreads thinly across all remaining PCs. PCA therefore acts as a **denoiser**.


In [ ]:
sc.pp.pca(adata, n_comps=50, svd_solver='arpack')

# The "elbow" plot: how much variance does each PC carry?
sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)

# Cells in PC1-PC2 space, colored by well-known marker genes (taken from adata.raw)
sc.pl.pca(adata, color=['CST3', 'NKG7', 'MS4A1', 'PPBP'], ncols=4)

# Which genes drive the first PCs?
sc.pl.pca_loadings(adata, components=[1, 2, 3])

#### **Part 6: The $k$-Nearest Neighbor Graph (Approximating the Manifold)**

A manifold is only *locally* Euclidean. Distances between nearby cells are meaningful, but long straight-line distances "cut through" regions where no cells exist. We therefore keep only **local** information. For every cell, we connect it to its $k$ closest cells in PCA space (here $k = 10$, using the first 40 PCs).

Scanpy stores two sparse $n \times n$ matrices:
*   `adata.obsp['distances']`: Euclidean distances to the $k-1$ nearest neighbors (the cell itself is excluded).
*   `adata.obsp['connectivities']`: Symmetric edge weights in $[0, 1]$, computed with UMAP's fuzzy-set formula. Clustering and embedding both run on this weighted graph.


In [ ]:
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)

conn = adata.obsp['connectivities']
dist = adata.obsp['distances']

print(f"Graph: {adata.n_obs:,} nodes")
print(f"Undirected edges: {conn.nnz // 2:,}")
print(f"Mean degree (after symmetrization): {conn.nnz / adata.n_obs:.1f}")
print(f"Neighbors stored per cell in 'distances': {np.unique(np.diff(dist.tocsr().indptr))}")
print(f"Sparsity of the graph: {100 * (1 - conn.nnz / adata.n_obs**2):.3f}% zeros")


#### **Part 7: Leiden Community Detection and UMAP**

**Leiden** looks for a partition of the graph that maximizes **modularity**: more edge weight *inside* communities than we would expect by chance.

$$ Q = \frac{1}{2m} \sum_{ij} \left( A_{ij} - \gamma \frac{k_i k_j}{2m} \right) \delta(c_i, c_j) $$

Here $A_{ij}$ is the edge weight, $k_i$ is the weighted degree of cell $i$, $m$ is the total edge weight, and $\delta(c_i, c_j)=1$ when the two cells belong to the same community. The **resolution** $\gamma$ is the knob that controls granularity: larger $\gamma$ gives more, smaller clusters. Leiden improves on the older Louvain algorithm because it guarantees that every community is internally connected.

**UMAP** then lays the same graph out in 2D for visualization. Keep this in mind: clusters are computed on the **graph**, *not* on the UMAP picture.


In [ ]:
sc.tl.leiden(adata, resolution=0.9, random_state=0,
             flavor='igraph', n_iterations=2, directed=False)
sc.tl.umap(adata, random_state=0)

print(f"Leiden found {adata.obs['leiden'].nunique()} clusters")
print(adata.obs['leiden'].value_counts().sort_index().to_string())

sc.pl.umap(adata, color=['leiden'], legend_loc='on data', title='Leiden clusters (resolution 0.9)')

# Show the k-NN graph edges underneath the embedding
sc.pl.umap(adata, color='leiden', edges=True, edges_width=0.05, size=15,
           title='k-NN graph edges drawn on the UMAP layout')

#### **Part 8: Marker Genes and Immune Lineage Determination**

Clusters are just numbers until we attach biology to them. We take two complementary approaches:
1.  **Data-driven:** For each cluster, find the genes most significantly up-regulated against all other cells (Wilcoxon rank-sum test).
2.  **Knowledge-driven:** Score each cell against curated **marker gene sets** for known PBMC lineages. Then give each cluster the label with the highest mean score.

| Lineage | Classic markers |
|---|---|
| CD4 T cells | `IL7R`, `CCR7`, `LDHB` |
| CD8 T cells | `CD8A`, `CD8B`, `GZMK` |
| NK cells | `GNLY`, `NKG7`, `KLRD1` |
| B cells | `MS4A1`, `CD79A`, `CD79B` |
| CD14⁺ Monocytes | `CD14`, `LYZ`, `S100A8`, `S100A9` |
| FCGR3A⁺ (CD16⁺) Monocytes | `FCGR3A`, `MS4A7`, `LST1` |
| Dendritic cells | `FCER1A`, `CST3`, `CLEC10A` |
| Megakaryocytes / Platelets | `PPBP`, `PF4` |

In [ ]:
# Data-driven: differential expression, one cluster vs the rest
sc.tl.rank_genes_groups(adata, 'leiden', method='wilcoxon')
sc.pl.rank_genes_groups(adata, n_genes=15, sharey=False)

# Top 5 genes per cluster as a table
top_genes = pd.DataFrame(adata.uns['rank_genes_groups']['names']).head(5)
top_genes

In [ ]:
marker_genes = {
    'CD4 T':        ['IL7R', 'CCR7', 'LDHB'],
    'CD8 T':        ['CD8A', 'CD8B', 'GZMK'],
    'NK':           ['GNLY', 'NKG7', 'KLRD1'],
    'B':            ['MS4A1', 'CD79A', 'CD79B'],
    'CD14 Mono':    ['CD14', 'LYZ', 'S100A8', 'S100A9'],
    'FCGR3A Mono':  ['FCGR3A', 'MS4A7', 'LST1'],
    'Dendritic':    ['FCER1A', 'CST3', 'CLEC10A'],
    'Megakaryocyte':['PPBP', 'PF4'],
}

# Dot plot: dot size = fraction of cells expressing, color = mean expression
sc.pl.dotplot(adata, marker_genes, groupby='leiden', standard_scale='var')

# Knowledge-driven annotation: score every cell for every gene set
score_cols = []
for cell_type, genes in marker_genes.items():
    col = f'score_{cell_type}'
    sc.tl.score_genes(adata, gene_list=genes, score_name=col, use_raw=True)
    score_cols.append(col)

# Mean score per cluster -> pick the best-matching lineage
cluster_scores = adata.obs.groupby('leiden', observed=True)[score_cols].mean()
cluster_to_type = cluster_scores.idxmax(axis=1).str.replace('score_', '', regex=False).to_dict()

adata.obs['cell_type'] = adata.obs['leiden'].astype(str).map(cluster_to_type).astype('category')

print("Cluster -> cell type assignment:")
for cl, ct in cluster_to_type.items():
    print(f"  cluster {cl:>2}: {ct}")

sc.pl.umap(adata, color='cell_type', legend_loc='on data', legend_fontsize=8,
           title='Annotated PBMC lineages')

#### **Part 9: Coarse-Grained Topology (PAGA)**

UMAP is a nice picture, but it can mislead us about how clusters relate globally. **PAGA** (Partition-based Graph Abstraction) collapses the $k$-NN graph to one node per cell type. Two nodes are joined by an edge whose weight measures how much *more* the two groups are connected than we would expect by chance. The result is a map of which lineages lie next to each other on the manifold.


In [ ]:
sc.tl.paga(adata, groups='cell_type')
sc.pl.paga(adata, threshold=0.05, fontsize=9, node_size_scale=2,
           title='PAGA: connectivity between lineages')

#### **Part 10: Continuous State Transitions (Diffusion Pseudotime)**

Not every cell falls cleanly into a category. Classical **CD14⁺ monocytes** gradually mature into **CD16⁺ (FCGR3A⁺) monocytes**, and cells caught in the middle of this transition sit *between* the two clusters.

**Diffusion pseudotime (DPT)** models a random walk on the $k$-NN graph and measures how far each cell is from a chosen **root** cell. We place the root at the most "classical" monocyte. Pseudotime then orders the cells along the maturation trajectory, and we can follow marker genes rising and falling along it.


In [ ]:
mono_types = [t for t in ['CD14 Mono', 'FCGR3A Mono'] if t in adata.obs['cell_type'].cat.categories]
mono = adata[adata.obs['cell_type'].isin(mono_types)].copy()
print(f"Monocytes used for trajectory: {mono.n_obs} cells ({', '.join(mono_types)})")

# Rebuild the graph on the monocyte subset only, then compute a diffusion map
sc.pp.neighbors(mono, n_neighbors=15, n_pcs=20)
sc.tl.diffmap(mono)

# Root = the most 'classical' CD14+ monocyte
classical_score = mono.obs['score_CD14 Mono'].values
if 'score_FCGR3A Mono' in mono.obs:
    classical_score = classical_score - mono.obs['score_FCGR3A Mono'].values
mono.uns['iroot'] = int(np.argmax(classical_score))

sc.tl.dpt(mono)
mono.obs['dpt_pseudotime'] = mono.obs['dpt_pseudotime'].replace(np.inf, np.nan)

sc.pl.umap(mono, color=['cell_type', 'dpt_pseudotime'], size=40)

# Marker dynamics along pseudotime
expr = sc.get.obs_df(mono, keys=['CD14', 'LYZ', 'FCGR3A', 'MS4A7', 'dpt_pseudotime'], use_raw=True)
expr = expr.dropna().sort_values('dpt_pseudotime')

plt.figure(figsize=(8, 4))
for gene, color in zip(['CD14', 'LYZ', 'FCGR3A', 'MS4A7'], ['tab:blue', 'tab:cyan', 'tab:red', 'tab:orange']):
    smooth = expr[gene].rolling(window=50, center=True, min_periods=10).mean()
    plt.plot(expr['dpt_pseudotime'], smooth, label=gene, color=color, linewidth=2)
plt.xlabel('Diffusion pseudotime')
plt.ylabel('log-normalized expression (rolling mean)')
plt.title('Classical → non-classical monocyte transition')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


#### **Part 11: A Data-Driven "Waddington Landscape"**

To close the loop with Waddington's metaphor, we turn cell density in the UMAP plane into a **pseudo-potential**:

$$ U(x, y) = -\log \hat{p}(x, y) $$

where $\hat{p}$ is a kernel density estimate. Densely populated, stable cell states turn into **valleys**. Sparsely populated transition zones turn into **ridges**.

*This is only an illustration. UMAP distances are not physically meaningful, and PBMCs are mostly mature cells, not cells that are actively differentiating. Even so, the picture captures the central intuition: cell types are attractor basins in gene-expression space.*


In [ ]:
import plotly.graph_objects as go
from scipy.stats import gaussian_kde

coords = adata.obsm['X_umap']
kde = gaussian_kde(coords.T, bw_method=0.12)

pad = 1.0
xg = np.linspace(coords[:, 0].min() - pad, coords[:, 0].max() + pad, 120)
yg = np.linspace(coords[:, 1].min() - pad, coords[:, 1].max() + pad, 120)
XX, YY = np.meshgrid(xg, yg)
density = kde(np.vstack([XX.ravel(), YY.ravel()])).reshape(XX.shape)
potential = -np.log(density + 1e-4)

fig = go.Figure(data=[go.Surface(z=potential, x=XX, y=YY, colorscale='Viridis', opacity=0.85)])

# Label each valley with its cell type
annotations = []
for ct in adata.obs['cell_type'].cat.categories:
    centroid = coords[adata.obs['cell_type'] == ct].mean(axis=0)
    z = -np.log(kde(centroid.reshape(2, 1))[0] + 1e-4)

    annotations.append(dict(
        x=centroid[0],
        y=centroid[1],
        z=z + 0.5,
        text=ct,
        showarrow=False,
        font=dict(color="black", size=12)
    ))

fig.update_layout(
    title='Data-driven "Waddington landscape" of PBMC states',
    scene=dict(
        xaxis_title='UMAP1',
        yaxis_title='UMAP2',
        zaxis_title='Pseudo-potential  −log density',
        annotations=annotations
    ),
    width=900,
    height=700,
)
fig.show()


### **Homework: Cellular State Spaces**

Use the concepts and tools demonstrated above to analyze the processed `adata` object.

#### **Task 1: PCA from Scratch**

Write a pure NumPy function `pca_svd(X, n_comps)` that:
1. Centers the matrix column-wise.
2. Computes the SVD with `np.linalg.svd(..., full_matrices=False)`.
3. Returns the cell **scores** (`n_cells × n_comps`) and the **explained variance ratio** of each component (relative to the *total* variance).

Then run it on `adata.X` and check your result against Scanpy for the first 5 PCs:
*   The absolute Pearson correlation between your PC$_i$ and `adata.obsm['X_pca'][:, i]` should be ≈ 1. (Why only the *absolute* value?)
*   Your explained variance ratios should match `adata.uns['pca']['variance_ratio']`.

In [ ]:
def pca_svd(X, n_comps=50):
    """
    Returns (scores, explained_variance_ratio) for the first n_comps principal components.
    """
    # ---- YOUR CODE HERE ----
    return None, None

#### **Task 2: How Many PCs?**

Using `adata.uns['pca']['variance_ratio']` (50 PCs), compute the cumulative explained variance.
1. How many PCs are needed to capture **80 %** of the variance explained by all 50 PCs?
2. Plot the cumulative curve, mark that point, and compare it with the `n_pcs=40` we used for the neighbor graph.


In [ ]:
# ---- YOUR CODE HERE ----

#### **Task 3: Build the $k$-NN Graph Yourself**

Using `sklearn.neighbors.NearestNeighbors` on the first 40 PCs (`adata.obsm['X_pca'][:, :40]`), find the 10 nearest neighbors of every cell. Remember that the first neighbor returned is the cell itself.

Compare your neighbor sets with those stored by Scanpy in `adata.obsp['distances']` (row `i` → `.indices` gives the neighbors of cell `i`). Report the **mean fraction of overlapping neighbors** across all cells.


In [ ]:
# ---- YOUR CODE HERE ----

#### **Task 4: Resolution Sweep**

Run Leiden at resolutions `[0.1, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0]` and store each result under its own key (`key_added=f'leiden_r{r}'`). Use the same settings as in Part 7 (`flavor='igraph', n_iterations=2, directed=False, random_state=0`).
1. Plot the number of clusters against the resolution.
2. Show side-by-side UMAPs for the lowest and highest resolution. Which lineages merge at low resolution? Which lineages split at high resolution?


In [ ]:
# ---- YOUR CODE HERE ----

#### **Task 5: The Mystery Cluster**

1. Find the **smallest** Leiden cluster (from the default `adata.obs['leiden']`).
2. Use `sc.get.rank_genes_groups_df(adata, group=...)` to list its top 10 marker genes.
3. Based on these genes, which cell type is it? Does it agree with the automatic `cell_type` label?
4. Finally, report the percentage of each annotated `cell_type` in the sample as a bar chart.


In [ ]:
# ---- YOUR CODE HERE ----

#### **Task 6: Interpretation Questions**

Answer the following in a text cell:

1. Why do we build the $k$-NN graph in a 40-dimensional PCA space instead of directly on ~2,000 highly variable genes (or on all ~13,000 genes)?
2. Why should clustering be done on the $k$-NN graph rather than on the 2D UMAP coordinates? What can (and cannot) be concluded from the distance between two clusters on a UMAP plot?
3. What happens to the PCA if we skip `sc.pp.scale` (z-scoring of genes)? Which genes would dominate PC1?
4. The resolution parameter $\gamma$ has no "correct" value. How would you decide on a resolution for a real study?
5. How does Waddington's epigenetic landscape map onto the objects we computed (clusters, graph edges, pseudotime)? Why is the classical → non-classical monocyte transition a better example of a *continuous* state change than, for example, B cells vs T cells?
6. If you change `random_state` in `sc.tl.umap`, the picture changes, but the Leiden clusters do not. Why?
